In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import pandas as pd
import xarray as xr
from datetime import datetime, timezone
from glob import glob
import pandas as pd
import osmnx
import geopandas as gpd
import geodatasets
import rioxarray
import xarray
from shapely.geometry import box, Point, LineString
import datashader as ds
import contextily as cx
import streamlit as st
from shapely import geometry

In [ ]:
project_dir = '.CSES_files'

print(f"Percorso cartella di progetto: {project_dir}")

EFD1 = 'CSES_data/CSES_01_EFD_1_L02_A1_213330_20211206_164953_20211206_172707_000.h5'
HEP1 = 'CSES_data/CSES_01_HEP_1_L02_A4_176401_20210407_182209_20210407_190029_000.h5'
HEP4 = 'CSES_data/CSES_01_HEP_4_L02_A4_202091_20210923_184621_20210923_192441_000.h5'
LAP1 = 'CSES_data/CSES_01_LAP_1_L02_A3_174201_20210324_070216_20210324_073942_000.h5'
SCM1 = 'CSES_data/CSES_01_SCM_1_L02_A2_183380_20210523_154551_20210523_162126_000.h5'
HEPD = 'CSES_data/CSES_HEP_DDD_0219741_20220117_214156_20220117_230638_L3_0000267631.h5'

file_list = [EFD1, HEP1, HEP4, LAP1, SCM1, HEPD]

def dataset(path):
    return xarray.open_dataset(path, engine = 'h5netcdf', phony_dims = 'sort')

def variables(data):
    return list(data.keys())

In [ ]:
CSES_DATA_TABLE = {'EFD':{'1':'ULF','2':'ELF','3':'VLF','4':'HF'},\
                   'HPM':{'1':'FGM1','2':'FGM2','3':'CDSM','5':'FGM1Hz'},\
                   'SCM':{'1':'ULF','2':'ELF','3':'VLF'},\
                   'LAP':{'1':'50mm', '2':'10mm'},\
                   'PAP':{'0':''}, \
                   'HEP':{'1':'P_L','2':'P_H','3':'D','4':'P_X'}}

# splits file name

def parse(filename):

    fl_list = filename[10:].split('_')
    out={}
    if len(filename[10:]) == 66:
        out['Satellite'] = fl_list[0]+fl_list[1]
        out['Instrument'] = fl_list[2]
        try:
            out['Data Product'] = CSES_DATA_TABLE[fl_list[2]][fl_list[3]]
        except:
            out['Data Product'] = 'Unknown' 
        out['Instrument No.'] = fl_list[3]
        out['Data Level'] = fl_list[4]
        out['orbitn'] = fl_list[6]
        out['year'] = fl_list[7][0:4]
        out['month'] = fl_list[7][4:6]
        out['day'] = fl_list[7][6:8]
        out['time'] = fl_list[8][0:2]+':'+fl_list[8][2:4]+':'+fl_list[8][4:6]
        out['t_start'] = datetime(int( out['year']),int(out['month']),int(out['day']),\
                            int(fl_list[8][0:2]),int(fl_list[8][2:4]),int(fl_list[8][4:6])) 
        out['t_end'] = datetime(int(fl_list[9][0:4]),int(fl_list[9][4:6]),int(fl_list[9][6:8]),\
                            int(fl_list[10][0:2]),int(fl_list[10][2:4]),int(fl_list[10][4:6]))
    elif len(filename[10:]) == 69:
        out['Satellite'] = fl_list[0]+'_01'
        out['Instrument'] = fl_list[1]
        out['Data Product'] = fl_list[2]
        out['Data Level'] = fl_list[-2]
        out['orbitn'] = fl_list[3]
        out['year'] = fl_list[4][0:4]
        out['month'] = fl_list[4][4:6]
        out['day'] = fl_list[4][6:8]
        out['time'] = fl_list[5][0:2]+':'+fl_list[5][2:4]+':'+fl_list[5][4:6]
        out['t_start'] = datetime(int( out['year']),int(out['month']),int(out['day']),\
                            int(fl_list[5][0:2]),int(fl_list[5][2:4]),int(fl_list[5][4:6])) 
        out['t_end'] = datetime(int(fl_list[6][0:4]),int(fl_list[6][4:6]),int(fl_list[6][6:8]),\
                            int(fl_list[7][0:2]),int(fl_list[7][2:4]),int(fl_list[7][4:6]))

    return out

In [ ]:
def extract_satellite_number(file_name):
    try:
        parts = file_name.split('_')
        satellite_number = parts[1]
        return satellite_number
    except IndexError:
        print(f"Errore nell'estrazione del numero del satellite per il file {file_name}")
        return None

In [ ]:
def extract_instrument_code(file_name):
    try:
        parts = file_name.split('_')
        instrument_code = parts[2]
        return instrument_code
    except IndexError:
        print(f"Errore nell'estrazione del codice strumento per il file {file_name}")
        return None

In [ ]:
def extract_instrument_number(file_name):
    try:
        parts = file_name.split('_')
        instrument_number = parts[3]
        return instrument_number
    except IndexError:
        print(f"Errore nell'estrazione del numero strumento per il file {file_name}")
        return None

In [ ]:
def extract_data_level(file_name):
    try:
        parts = file_name.split('_')
        data_level = parts[4]
        return data_level
    except IndexError:
        print(f"Errore nell'estrazione del livello dei dati per il file {file_name}")
        return None

In [ ]:
def extract_orbit(file_name):
    try:
        base_name = os.path.basename(file_name)
        parts = base_name.split('_')
        start_index = None
        for i in range(len(parts)):
            if parts[i].isdigit() and len(parts[i]) == 8: 
                start_index = i
                break
    
        if start_index is None:
            raise ValueError(f"Formato data non trovato nel nome del file: {file_name}")
        
        orbit = parts[start_index - 1]  
        return orbit
    except ValueError as e:
        print(f"Errore nel parsing dell'orbita per il file {file_name}: {e}")
        return None

In [ ]:
extract_orbit(EFD1)

In [ ]:
def extract_dates(file_name):
    try:
        base_name = os.path.basename(file_name) #returns the final component of a pathname
        parts = base_name.split('_')
        
        #find the index of the part that contains the start_date
        start_index = None
        for i in range(len(parts)):
            if parts[i].isdigit() and len(parts[i]) == 8:  # find the part with data format YYYYMMDD
                start_index = i
                break
        
        if start_index is None:
            raise ValueError(f"Formato data non trovato nel nome del file: {file_name}")
        
        start_date_str = '_'.join(parts[start_index:start_index + 2]) 
        end_date_str = '_'.join(parts[start_index + 2:start_index + 4])  
        
        start_date = datetime.strptime(start_date_str, '%Y%m%d_%H%M%S')
        end_date = datetime.strptime(end_date_str, '%Y%m%d_%H%M%S')
        
        return start_date, end_date
    except ValueError as e:
        print(f"Errore nel parsing delle date per il file {file_name}: {e}")
        return None, None

In [ ]:
extract_dates(EFD1)

In [ ]:
def parse_filename(file_name):
    satellite_nr = extract_satellite_number(file_name)
    instrument_code = extract_instrument_code(file_name)
    instrument_nr = extract_instrument_number(file_name)
    data_l = extract_data_level(file_name)
    start_date, end_date = extract_dates(file_name)
    semiorbit_nr = extract_orbit(file_name)
    return {
        'file_name': file_name,
        "satellite_nr": satellite_nr,
        "instrument_code": instrument_code,
        "instrument_nr": instrument_nr,
        "data_l":data_l,
        "semiorbit_nr": semiorbit_nr,
        "start_date": start_date, 
        "end_date" : end_date
    }

In [ ]:
data = []

for file in file_list:
    metadata = parse_filename(file)
    if metadata:
        data.append(metadata)
    #metadata["semiorbit_nr"]
    #semiorbits_geo[metadata["semiorbit_nr"]]
    # {
    #     "start_date": ...
    #     "start_date": ...
    #     "start_date": ...
    # }
if data:
    columns = list(data[0].keys())
else:
    columns = []


df = pd.DataFrame(data, columns=columns)

df

In [ ]:
def orbit_longitude(df, orbit):
    data = dataset(df)
    
    filtered = []
    
    for i in data:
        if (extract_orbit(i) == orbit):
            filtered.add(i)

    return filtered



In [ ]:
def find_file_by_orbit(df, orbit_nr):
    return df[df['orbit_nr'] == orbit_nr]


# We will use th h5py package for read the file and estract the coordinates
def extract_coordinates(file_name):
    with h5py.File(file_name, 'r') as f:
        latitudes = f['GEO_LAT'][:]
        longitudes = f['GEO_LON'][:]
        coordinates = np.column_stack((longitudes, latitudes))
    return coordinates


# filtering the coordinates
def filter_coordinates_in_bbox(coords, bbox):
    points_in_box = [coord for coord in coords if bbox.contains(Point(coord))]
    for coord in points_in_box:
        print (coord)
    return points_in_box


bbox = box(-55.85, -69.4844, -106.0156, 69.5816 )

orbit_coords = extract_coordinates(file_list[3])
print(orbit_coords)

filtered_coords = filter_coordinates_in_bbox(orbit_coords, bbox)


In [ ]:
def orbit_polygon(points, data):
    
    ds = dataset(data)

    longitudes = [point[0] for point in points]
    
    lon_min = min(longitudes)
    lon_max = max(longitudes)

    # .any(), .all()
    if((ds.GEO_LON.all() < lon_max) and (ds.GEO_LON.all() > lon_min)):
        return(ds)

polygon_points = [(100.0, 30.0), (120.0, 30.0), (120.0, 50.0), (100.0, 50.0)]

# # Test the polygon function with a polygon and one of the data files
filtered_points = orbit_polygon(polygon_points, file_list[0])


In [ ]:
def filter_polygon(points, orbits, dataset):
    
    geo_lat = dataset.GEO_LAT
    latitudes = [point[1] for point in points]
    lat_min = min(latitudes)
    lat_max = max(latitudes)

    lat_mask = (geo_lat >= lat_min) & (geo_lat <= lat_max)

    filtered = dataset.where(lat_mask, drop = True)

    return filtered 

In [ ]:
# extract times from YYYYMMDDHHMMSSmsmsms
def convert_time(date, request):
    if (request == 'year'): return date[:4]
    if (request == 'month'):    return date[4:6]
    if (request == 'day'):  return date[6:8]
    if (request == 'hour'): return date[8:10]
    if (request == 'minute'):   return date[10:12]
    if (request == 'second'):   return date[12:14]
    if (request == 'milisecond'):   return date[14:]


In [ ]:
def extract_dates(file_name):
    try:
        base_name = os.path.basename(file_name) #returns the final component of a pathname
        parts = base_name.split('_')
        
        #find the index of the part that contains the start_date
        start_index = None
        for i in range(len(parts)):
            if parts[i].isdigit() and len(parts[i]) == 8:  # find the part with data format YYYYMMDD
                start_index = i
                break
        
        if start_index is None:
            raise ValueError(f"Formato data non trovato nel nome del file: {file_name}")

        start_date_str = '_'.join(parts[start_index:start_index + 2]) 
        end_date_str = '_'.join(parts[start_index + 2:start_index + 4])  
        
        start_date = datetime.strptime(start_date_str, '%Y%m%d_%H%M%S')
        end_date = datetime.strptime(end_date_str, '%Y%m%d_%H%M%S')
        
        final = start_date, end_date
        return final
    except ValueError as e:
        print(f"Errore nel parsing delle date per il file {file_name}: {e}")
        return None, None

def extract_start_date(file_name):
    return (extract_dates(file_name)[0])

def extract_end_date(file_name):
    return (extract_dates(file_name)[1])

# input times as datetime
def filter_time(start, end, df):
    matching_files = df[(df['start_date'] <= start) & (df['end_date'] >= end)]
    return matching_files
    